In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
!pip install bitsandbytes

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [ ]:
!pip uninstall -y torchao
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 35.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import random
from pathlib import Path

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
from datasets import load_dataset


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(dataset)
print(dataset.column_names)
print(dataset[0])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})
['instruction', 'context', 'response', 'category']
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [ ]:
(dataset[4])

{'instruction': 'When was Tomoaki Komorida born?',
 'context': 'Komorida was born in Kumamoto Prefecture on July 10, 1981. After graduating from high school, he joined the J1 League club Avispa Fukuoka in 2000. Although he debuted as a midfielder in 2001, he did not play much and the club was relegated to the J2 League at the end of the 2001 season. In 2002, he moved to the J2 club Oita Trinita. He became a regular player as a defensive midfielder and the club won the championship in 2002 and was promoted in 2003. He played many matches until 2005. In September 2005, he moved to the J2 club Montedio Yamagata. In 2006, he moved to the J2 club Vissel Kobe. Although he became a regular player as a defensive midfielder, his gradually was played less during the summer. In 2007, he moved to the Japan Football League club Rosso Kumamoto (later Roasso Kumamoto) based in his local region. He played as a regular player and the club was promoted to J2 in 2008. Although he did not play as much, he

In [ ]:
DATA_DIR = Path("data/dolly_sft")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_MESSAGE = "You are a helpful assistant."

def build_user_message(instruction, context):
    instruction = (instruction or "").strip()
    context = (context or "").strip()

    if context:
        return f"Context:\n{context}\n\nInstruction:\n{instruction}"
    return instruction

def convert_example(example):
    user_content = build_user_message(
        example.get("instruction", ""),
        example.get("context", "")
    )

    assistant_content = (example.get("response", "") or "").strip()

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ],
        "category": example.get("category", "unknown")
    }

In [ ]:
raw = load_dataset("databricks/databricks-dolly-15k", split="train")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
raw = raw.shuffle(seed=42)

N_TRAIN = 1000
N_VALID = 100
N_TEST = 100
N_TOTAL = N_TRAIN + N_VALID + N_TEST

small = raw.select(range(N_TOTAL))

converted = [convert_example(ex) for ex in small]

train_data = converted[:N_TRAIN]
valid_data = converted[N_TRAIN:N_TRAIN + N_VALID]
test_data = converted[N_TRAIN + N_VALID:N_TOTAL]

print(len(train_data), len(valid_data), len(test_data))
(train_data[0])

1000 100 100


{'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'},
  {'role': 'user',
   'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?'},
  {'role': 'assistant',
   'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree'}],
 'category': 'open_qa'}

In [ ]:
def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_data, DATA_DIR / "train.jsonl")
save_jsonl(valid_data, DATA_DIR / "valid.jsonl")
save_jsonl(test_data, DATA_DIR / "test.jsonl")

print("Saved:")
print(DATA_DIR / "train.jsonl")
print(DATA_DIR / "valid.jsonl")
print(DATA_DIR / "test.jsonl")

Saved:
data/dolly_sft/train.jsonl
data/dolly_sft/valid.jsonl
data/dolly_sft/test.jsonl


In [ ]:
random.seed(42)

def make_eval_item(example):
    messages = example["messages"]

    assert messages[-1]["role"] == "assistant"

    return {
        "prompt_messages": messages[:-1],
        "target_answer": messages[-1]["content"],
        "category": example.get("category", "unknown")
    }

eval_50 = [make_eval_item(ex) for ex in random.sample(test_data, 50)]

save_jsonl(eval_50, DATA_DIR / "eval_50.jsonl")

print("Saved eval_50:", DATA_DIR / "eval_50.jsonl")
(eval_50[1])

Saved eval_50: data/dolly_sft/eval_50.jsonl


{'prompt_messages': [{'role': 'system',
   'content': 'You are a helpful assistant.'},
  {'role': 'user',
   'content': 'Explain the best way to swing a golf iron compared to a golf driver.'}],
 'target_answer': 'When hitting an iron you want to make a downward striking motion in the grass. The driver is more of a sweeping motion through the ball. Overall, shorten the back swing on the driver and it will help gain control.',
 'category': 'creative_writing'}

In [ ]:
def read_jsonl(path, n=3):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(json.loads(line))
    return rows

for item in read_jsonl(DATA_DIR / "train.jsonl", n=3):
    print("=" * 80)
    for msg in item["messages"]:
        print(msg["role"].upper())
        print(msg["content"][:700])
        print()

SYSTEM
You are a helpful assistant.

USER
Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?

ASSISTANT
Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree

SYSTEM
You are a helpful assistant.

USER
Give me a list of basic ingredients for baking cookies

ASSISTANT
To bake cookies you will need flour, sugar, eggs, milk, butter, baking powder, and if you want; you can add chocolate chips, cinnamon, walnuts, or macadamia nuts.  By combining the ingredients you will form a soft dough that you can scoop onto a baking sheet and bake in the oven until cooked thoroughly through.

SYSTEM
You are a helpful assistant.

USER
Write a funny and whimsical horoscope reading

ASSISTANT
The stars say you should be patien

In [ ]:
def validate_jsonl(path):
    path = Path(path)
    seen = set()
    errors = []

    with open(path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            try:
                row = json.loads(line)
            except Exception as e:
                errors.append((line_idx, f"Invalid JSON: {e}"))
                continue

            if "messages" not in row:
                errors.append((line_idx, "Missing messages"))
                continue

            messages = row["messages"]

            if not isinstance(messages, list) or len(messages) < 3:
                errors.append((line_idx, "messages must be a list with at least 3 items"))
                continue

            roles = [m.get("role") for m in messages]
            if roles[0] != "system" or roles[1] != "user" or roles[-1] != "assistant":
                errors.append((line_idx, f"Bad roles order: {roles}"))

            for m in messages:
                if m.get("role") not in {"system", "user", "assistant"}:
                    errors.append((line_idx, f"Invalid role: {m.get('role')}"))
                if not str(m.get("content", "")).strip():
                    errors.append((line_idx, "Empty content"))

            key = json.dumps(messages, ensure_ascii=False)
            if key in seen:
                errors.append((line_idx, "Duplicate example"))
            seen.add(key)

    print(f"{path}:")
    print("Errors:", len(errors))

    for err in errors[:10]:
        print(err)

validate_jsonl(DATA_DIR / "train.jsonl")
validate_jsonl(DATA_DIR / "valid.jsonl")
validate_jsonl(DATA_DIR / "test.jsonl")

data/dolly_sft/train.jsonl:
Errors: 0
data/dolly_sft/valid.jsonl:
Errors: 0
data/dolly_sft/test.jsonl:
Errors: 0


In [ ]:
DATA_DIR = Path("data/dolly_sft")
EVAL_PATH = DATA_DIR / "eval_50.jsonl"

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
OUT_PATH = OUT_DIR / "base_qwen2_5_0_5b_eval_outputs.jsonl"

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
model.eval()

def generate_answer(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )
    return answer.strip()

In [ ]:
with open(EVAL_PATH, "r", encoding="utf-8") as f_in, open(OUT_PATH, "w", encoding="utf-8") as f_out:
    for idx, line in enumerate(f_in):
        item = json.loads(line)

        pred = generate_answer(item["prompt_messages"])

        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"Done {idx + 1}/50")

print(f"Saved outputs to: {OUT_PATH}")

Done 1/50
Done 2/50
Done 3/50
Done 4/50
Done 5/50
Done 6/50
Done 7/50
Done 8/50
Done 9/50
Done 10/50
Done 11/50
Done 12/50
Done 13/50
Done 14/50
Done 15/50
Done 16/50
Done 17/50
Done 18/50
Done 19/50
Done 20/50
Done 21/50
Done 22/50
Done 23/50
Done 24/50
Done 25/50
Done 26/50
Done 27/50
Done 28/50
Done 29/50
Done 30/50
Done 31/50
Done 32/50
Done 33/50
Done 34/50
Done 35/50
Done 36/50
Done 37/50
Done 38/50
Done 39/50
Done 40/50
Done 41/50
Done 42/50
Done 43/50
Done 44/50
Done 45/50
Done 46/50
Done 47/50
Done 48/50
Done 49/50
Done 50/50
Saved outputs to: outputs/base_qwen2_5_0_5b_eval_outputs.jsonl


In [ ]:
def preview_outputs(path, n=3):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break

            item = json.loads(line)

            print("=" * 100)
            print("ID:", item["id"])
            print("category:", item["category"])
            for msg in item["prompt_messages"]:
                print(f"{msg['role'].upper()}: {msg['content'][:700]}")

            print("\n target answer:")
            print(item["target_answer"][:1000])

            print("\n model answer:")
            print(item["model_answer"][:1000])

preview_outputs(OUT_PATH, n=3)

ID: 0
category: classification
SYSTEM: You are a helpful assistant.
USER: Classify these objects based on their shape.
wheel, coin, CD, stamp, chess board

 target answer:
Round - wheel, coin, CD
Square - stamp, chess board

 model answer:
Wheel: Shape of a circle
Coin: Shape of a round object
CD: Shape of a rectangular box
Stamp: Shape of a square or rectangle
Chess Board: Shape of an equilateral triangle
ID: 1
category: creative_writing
SYSTEM: You are a helpful assistant.
USER: Explain the best way to swing a golf iron compared to a golf driver.

 target answer:
When hitting an iron you want to make a downward striking motion in the grass. The driver is more of a sweeping motion through the ball. Overall, shorten the back swing on the driver and it will help gain control.

 model answer:
Swinging a golf iron versus a golf driver is a matter of personal preference and skill level, but there are some general tips that can help you get started.

Golf Iron:

1. Swing: A good swing for a

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

NameError: name 'model' is not defined

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
DATA_DIR = Path("data/dolly_sft")
TRAIN_PATH = str(DATA_DIR / "train.jsonl")
VALID_PATH = str(DATA_DIR / "valid.jsonl")

In [ ]:
OUTPUT_DIR = "outputs/qwen2_5_0_5b_dolly_qlora"

In [ ]:
OUTPUT_after_fine_tuning = DATA_DIR / "qwen2_5_0_5b_eval_outputsafter_fine_tuning.jsonl"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

In [ ]:
train_dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
eval_dataset = load_dataset("json", data_files=VALID_PATH, split="train")

In [ ]:
arg_train = SFTConfig(
    output_dir=OUTPUT_DIR,

    max_length=1024,
    packing=False,

    num_train_epochs=1,
    learning_rate=2e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,

    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",

    report_to="none",
    assistant_only_loss=True,
)

/tmp/ipykernel_13429/2982477940.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  arg_train = SFTConfig(


In [ ]:
# import gc, torch

# try:
#     del model
#     del trainer
# except:
#     pass

# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
print("CUDA:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("model dtype:", next(model.parameters()).dtype)

CUDA: Tesla T4
bf16 supported: True
model dtype: torch.float32


In [ ]:
# model = model.to(torch.float16)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=arg_train,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter to:", OUTPUT_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,2.106211,1.664776,1.643270,0.634533,158760.000000
63,1.901987,1.663001,1.640009,0.633493,197108.000000


Saved adapter to: outputs/qwen2_5_0_5b_dolly_qlora


In [ ]:
DATA_DIR = Path("data/dolly_sft")
EVAL_PATH = DATA_DIR / "eval_50.jsonl"

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FT_OUT_PATH = OUT_DIR / "ft_qwen2_5_0_5b_eval_outputs.jsonl"

In [ ]:
model = trainer.model
model.eval()

def generate_answer_after_ft(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
with open(EVAL_PATH, "r", encoding="utf-8") as f_in, open(FT_OUT_PATH, "w", encoding="utf-8") as f_out:
    for idx, line in enumerate(f_in):
        item = json.loads(line)

        pred = generate_answer_after_ft(item["prompt_messages"])

        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"Done {idx + 1}/50")

print(f"Saved fine-tuned outputs to: {FT_OUT_PATH}")

Done 1/50
Done 2/50
Done 3/50
Done 4/50
Done 5/50
Done 6/50
Done 7/50
Done 8/50
Done 9/50
Done 10/50
Done 11/50
Done 12/50
Done 13/50
Done 14/50
Done 15/50
Done 16/50
Done 17/50
Done 18/50
Done 19/50
Done 20/50
Done 21/50
Done 22/50
Done 23/50
Done 24/50
Done 25/50
Done 26/50
Done 27/50
Done 28/50
Done 29/50
Done 30/50
Done 31/50
Done 32/50
Done 33/50
Done 34/50
Done 35/50
Done 36/50
Done 37/50
Done 38/50
Done 39/50
Done 40/50
Done 41/50
Done 42/50
Done 43/50
Done 44/50
Done 45/50
Done 46/50
Done 47/50
Done 48/50
Done 49/50
Done 50/50
Saved fine-tuned outputs to: outputs/ft_qwen2_5_0_5b_eval_outputs.jsonl


In [ ]:
BASE_PATH = Path("outputs/base_qwen2_5_0_5b_eval_outputs.jsonl")
FT_PATH = Path("outputs/ft_qwen2_5_0_5b_eval_outputs.jsonl")

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

base_rows = read_jsonl(BASE_PATH)
ft_rows = read_jsonl(FT_PATH)

for i in range(5):
    base = base_rows[i]
    ft = ft_rows[i]

    print("=" * 100)
    print("ID:", i)
    print("category:", ft["category"])
    print()
    for msg in ft["prompt_messages"]:
        print(f"{msg['role'].upper()}: {msg['content']}")

    print("\nTarget:")
    print(ft["target_answer"])

    print("\nBase model answer:")
    print(base["model_answer"])

    print("\nFine Tuned model answer:")
    print(ft["model_answer"])

ID: 0
category: classification

SYSTEM: You are a helpful assistant.
USER: Classify these objects based on their shape.
wheel, coin, CD, stamp, chess board

Target:
Round - wheel, coin, CD
Square - stamp, chess board

Base model answer:
Wheel: Shape of a circle
Coin: Shape of a round object
CD: Shape of a rectangular box
Stamp: Shape of a square or rectangle
Chess Board: Shape of an equilateral triangle

Fine Tuned model answer:
Wheel: Shape is circular
Coin: Shape is flat
CD: Shape is rectangular
Stamp: Shape is round
Chess Board: Shape is square
ID: 1
category: creative_writing

SYSTEM: You are a helpful assistant.
USER: Explain the best way to swing a golf iron compared to a golf driver.

Target:
When hitting an iron you want to make a downward striking motion in the grass. The driver is more of a sweeping motion through the ball. Overall, shorten the back swing on the driver and it will help gain control.

Base model answer:
Swinging a golf iron versus a golf driver is a matter of 